## Analyze Me #2

#### Jaden Vongnarath

#### MIS 769 - Spring 2026

---

### Environment Setup

In [ ]:
# Install required packages

!uv pip install spacy nltk wordcloud gensim scikit-learn matplotlib numpy pandas seaborn sentence-transformers faiss-cpu torch bs4 -q
!python -m spacy download en_core_web_sm -q

print("✅ Libraries installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 54.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Libraries installed!


In [21]:
# Import required packages

import os
import re
import spacy
import random
import requests
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import gensim.downloader as api

from spacy import displacy
from bs4 import BeautifulSoup
from collections import Counter
from sklearn.manifold import TSNE
from gensim.models import Word2Vec
from IPython.display import display, HTML
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load("en_core_web_sm")

random.seed(103)
np.random.seed(103)

print("✅ Libraries imported!")

✅ Libraries imported!


In [22]:
# Function scrapes BEA press releases
def bea_release_scrape(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")
    
    # Retrieve title
    title = soup.find("h1").get_text(strip=True)
    
    # Retrieve main body text
    article = soup.find("article") or soup.find("div", class_="field-items")
    text = article.get_text(separator="\n", strip=True)
    
    # Clean text
    text = re.sub(r'\n{3,}', '\n\n', text) # remove extra newlines
    text = re.sub(r'\s{2,}', ' ', text) # remove extra spaces
    
    return {"title": title, "url": url, "text": text}

print("✅ Scraper created!")

✅ Scraper created!


In [26]:
# Scrape press release by inserting URL:
url = "https://www.bea.gov/news/2026/gdp-second-estimate-4th-quarter-and-year-2025"
releases = [bea_release_scrape(url)]
df = pd.DataFrame(releases)

print(f"✅ BEA press release [{url}] scraped!")

✅ BEA press release [https://www.bea.gov/news/2026/gdp-second-estimate-4th-quarter-and-year-2025] scraped!


In [27]:
os.makedirs("bea_releases", exist_ok=True)

for i, row in df.iterrows():
    filename = f"bea_releases/release_{i+1}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"TITLE: {row['title']}\n")
        f.write(f"URL: {row['url']}\n\n")
        f.write(row["text"])
        
print("✅ Press release saved to 'bea_releases' folder!")

✅ Press release saved to 'bea_releases' folder!


In [28]:
for i, row in df.iterrows():
    display(HTML(f"<h3>{row['title']}</h3><pre>{row['text']}</pre>"))
    
print("✅ Press release displayed in notebook!")

✅ Press release displayed in notebook!


In [49]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("=" * 60)
print("✅ MODEL LOADED")
print("=" * 60)
print(f"Model: all-MiniLM-L6-v2")
print(f"Embedding Dimension: {model.get_sentence_embedding_dimension()}")
print(f"Max sequence length: {model.max_seq_length}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ MODEL LOADED
Model: all-MiniLM-L6-v2
Embedding Dimension: 384
Max sequence length: 256


In [ ]:
print(f"✅ Loaded press}")